# Pipeline v2 — Stage A

This notebook is a thin interface for the complete Stage A implementation. Data access, cohort reconstruction, grouped CV, model fitting, artifact export, and smoke inference live in Python modules.

In [8]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != 'ML-SbPSTU-2026' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name != 'ML-SbPSTU-2026':
    raise RuntimeError('Run this notebook from inside the ML-SbPSTU-2026 repository')
RESEARCH_SCRIPTS_ROOT = PROJECT_ROOT / 'Scripts' / 'ResearchScripts'
if str(RESEARCH_SCRIPTS_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_SCRIPTS_ROOT))

import pandas as pd
from IPython.display import Markdown, display
from ml_tests.stage_a import run_stage_a

OUTPUT_ROOT = PROJECT_ROOT / 'Scripts' / 'ResearchScripts' / 'Outputs' / 'pipeline_v2'
PROFILE = 'final'
RUN_STAGE_A = False
OVERWRITE_V2_ARTIFACTS = False

## Execute the stage

Leave `RUN_STAGE_A = False` when reopening the notebook. Existing reports and artifacts can be inspected without recomputing embeddings or retraining models.

In [9]:
stage_result = None
if RUN_STAGE_A:
    stage_result = run_stage_a(profile=PROFILE, overwrite_artifacts=OVERWRITE_V2_ARTIFACTS)
stage_result

## Stage report

The report consolidates T00–T05, database fingerprints, cohort warnings, tests, metrics, and the five focused user-validation checks.

In [10]:
report_path = OUTPUT_ROOT / 'stages' / 'A' / 'STAGE_REPORT.md'
display(Markdown(report_path.read_text(encoding='utf-8'))) if report_path.exists() else print('Run Stage A first')

# Pipeline v2 — Stage A report

Status: **PASS — ready for user validation**

## Outcome

The complete ML stage was executed read-only against the forest database. Two versioned v2 artifacts were trained and four-artifact inference was smoke-tested.

## Task status

| Task | Status | Result | Evidence |
|---|---:|---|---|
| T00 | PASS | Frozen database, table-count and legacy-artifact inventory. | `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T00_baseline/baseline_inventory.json` |
| T01 | PASS | Frozen 502 human-VALID events; 271 are ready for both models; their model-ready exact-old subsets contain 135 RemoteCLIP and 127 XGBoost events. | `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T01_frozen_cohorts/full_cohort.csv` |
| T02 | PASS | Saved 50 grouped folds with hash b9cb126c8c6adbaba982b95b6f65ad6590188b52a2c4627e6bed333715946fa2. | `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T02_grouped_splits/group_splits_final.json` |
| T03 | PASS | Completed OLD/FULL CV on frozen folds and exported XGBoost v2. | `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T03_xgboost_v2/summary_metrics.csv` |
| T04 | PASS | Completed OLD/FULL CV using cached embeddings and exported RemoteCLIP-LogReg v2. | `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T04_remoteclip_v2/summary_metrics.csv` |
| T05 | PASS | Produced four independent predictions for 10 frozen events. | `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T05_four_artifact_smoke/smoke_predictions.csv` |

## Cohort audit

- Current human-VALID events: **502**.
- Current human-VALID samples: **232**.
- Common PRE/POST-ready events: **271**.
- Ground-truth sources: `{"EVENT_REVIEW": 138, "SAMPLE_MANUAL_VALIDATION": 364}`.
- Event-download links used: `{"explicit": 4213, "virtual_dominant_backfill": 1025}`.
- Exact RemoteCLIP old events mapped/current: **142**; model-ready on the common input set: **135**.
- Exact XGBoost old events mapped/current: **140**; model-ready on the common input set: **127**.
- Frozen quick split hash: `8e4794c20a888a57fd27a0ad296aaa5e95115edd74516ef73d1da45fe254ad76`.
- Frozen final split hash: `b9cb126c8c6adbaba982b95b6f65ad6590188b52a2c4627e6bed333715946fa2`.
- Selected split hash: `b9cb126c8c6adbaba982b95b6f65ad6590188b52a2c4627e6bed333715946fa2`.
- Database SHA before/after: `46c6c3c8e3734cec8dd6b2b08d8cc950c8d5093768119ee1ecb955da2c8ef053` / `46c6c3c8e3734cec8dd6b2b08d8cc950c8d5093768119ee1ecb955da2c8ef053`.

> [!IMPORTANT]
> FULL_COHORT combines authoritative event-level VALID reviews with older sample-level ManualValidation=Valid labels. An explicit EventReview always takes precedence, including exclusion of DRAFT/REJECTED events from the legacy source.

## Model metrics

| Model | Cohort | Recall Logging | Balanced accuracy | Macro F1 | Fold runs |
|---|---|---:|---:|---:|---:|
| XGBOOST | FULL_COHORT_CV | 0.719 | 0.794 | 0.797 | 50 |
| XGBOOST | OLD_COHORT_CV | 0.655 | 0.763 | 0.766 | 50 |
| REMOTECLIP_LOGREG | FULL_COHORT_CV | 0.979 | 0.950 | 0.944 | 50 |
| REMOTECLIP_LOGREG | OLD_COHORT_CV | 0.888 | 0.915 | 0.916 | 50 |

## Self-checks

- No `sample_id` occurs in train and test within a grouped fold.
- Every test fold contains both target classes.
- OLD and FULL modes use the same frozen test events.
- Legacy artifacts were not overwritten.
- Both v2 artifacts passed reload and finite-probability smoke tests.
- Four-artifact smoke table contains **10** event rows.
- Forest SQLite SHA-256 is unchanged.

## User validation — 5 focused checks

1. Open `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T01_frozen_cohorts/full_cohort.csv` and verify several human labels/review timestamps; inspect `virtual_event_backfill_plan.json` to confirm the legacy mapping stayed read-only.
2. Open `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T02_grouped_splits/fold_summary.csv`; `group_overlap` must be zero and both test class counts must be positive.
3. Compare OLD/FULL rows in `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T03_xgboost_v2/summary_metrics.csv` and `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T04_remoteclip_v2/summary_metrics.csv`.
4. Inspect `Scripts/ResearchScripts/Outputs/pipeline_v2/checkpoints/T05_four_artifact_smoke/smoke_predictions.csv`; every event has four finite predictions.
5. Inspect the v2 manifests under `Scripts/ResearchTool/Data/forest/models/*/v2/` and confirm v1 files are untouched.

## Artifacts

- Run directory: `Scripts/ResearchScripts/Outputs/pipeline_v2/stage_a_runs/20261116_002033`.
- Plot: `Scripts/ResearchScripts/Outputs/pipeline_v2/stage_a_runs/20261116_002033/model_comparison.png`.
- Machine-readable summary: `Scripts/ResearchScripts/Outputs/pipeline_v2/stages/A/stage_summary.json`.

## Known limitation / refactoring candidate

- `EventReviewRevision` does not exist before Stage B, therefore the frozen manifest stores a deterministic `review_snapshot_id` derived from the current review row and `updated_at`.
- Older sample-level labels are inherited by linked events because event-level reviews did not exist for that historical cohort. Explicit event reviews always override that inherited label.


## Frozen cohort and grouped splits

The first table is the immutable human-reviewed event cohort. The second verifies class balance and zero sample leakage in every outer fold.

In [11]:
cohort_dir = OUTPUT_ROOT / 'checkpoints' / 'T01_frozen_cohorts'
cohort_path = cohort_dir / 'full_cohort.csv'
cohort_audit_path = cohort_dir / 'cohort_audit.json'
remoteclip_old_path = cohort_dir / 'remoteclip_legacy_cohort.csv'
xgboost_old_path = cohort_dir / 'xgboost_legacy_cohort.csv'
split_path = OUTPUT_ROOT / 'checkpoints' / 'T02_grouped_splits' / 'fold_summary.csv'
split_manifest_path = OUTPUT_ROOT / 'checkpoints' / 'T02_grouped_splits' / 'split_manifest.json'
cohort_df = pd.read_csv(cohort_path) if cohort_path.exists() else pd.DataFrame()
cohort_audit = json.loads(cohort_audit_path.read_text(encoding='utf-8')) if cohort_audit_path.exists() else {}
remoteclip_old_df = pd.read_csv(remoteclip_old_path) if remoteclip_old_path.exists() else pd.DataFrame()
xgboost_old_df = pd.read_csv(xgboost_old_path) if xgboost_old_path.exists() else pd.DataFrame()
split_df = pd.read_csv(split_path) if split_path.exists() else pd.DataFrame()
split_manifest = json.loads(split_manifest_path.read_text(encoding='utf-8')) if split_manifest_path.exists() else {}
cohort_summary = pd.DataFrame([{
    'events': len(cohort_df),
    'samples': cohort_df['sample_id'].nunique() if not cohort_df.empty else 0,
    'logging': int((cohort_df['human_label'] == 'Logging').sum()) if not cohort_df.empty else 0,
    'wildfire': int((cohort_df['human_label'] == 'Wildfire').sum()) if not cohort_df.empty else 0,
    'event_review_events': cohort_audit.get('ground_truth_source_counts', {}).get('EVENT_REVIEW', 0),
    'legacy_manual_events': cohort_audit.get('ground_truth_source_counts', {}).get('SAMPLE_MANUAL_VALIDATION', 0),
    'explicit_event_links': cohort_audit.get('event_link_counts', {}).get('explicit', 0),
    'virtual_legacy_links': cohort_audit.get('event_link_counts', {}).get('virtual_dominant_backfill', 0),
    'remoteclip_old_ready_events': cohort_audit.get('remoteclip_old_events', 0),
    'xgboost_old_ready_events': cohort_audit.get('xgboost_old_events', 0),
}])
not_ready_df = cohort_df[(cohort_df['xgboost_input_status'] != 'READY') | (cohort_df['remoteclip_input_status'] != 'READY')] if not cohort_df.empty else cohort_df
duplicate_ids_df = cohort_df[cohort_df.duplicated('record_id', keep=False)] if not cohort_df.empty else cohort_df
duplicate_sample_year_df = cohort_df[cohort_df.duplicated(['sample_id', 'event_year'], keep=False)] if not cohort_df.empty else cohort_df
legacy_overlap_df = pd.DataFrame([cohort_audit.get('legacy_current_valid_overlap', {})])
display(Markdown('### Cohort counts'))
display(cohort_summary)
display(Markdown('### Excluded from the common model-ready cohort'))
display(not_ready_df[['event_id', 'sample_id', 'event_year', 'human_label', 'xgboost_input_status', 'remoteclip_input_status']])
display(Markdown('### Duplicate audit'))
display(pd.DataFrame([{'duplicate_record_ids': len(duplicate_ids_df), 'duplicate_sample_year_rows': len(duplicate_sample_year_df)}]))
display(Markdown('### Exact legacy-to-current VALID overlap'))
display(legacy_overlap_df)
display(Markdown('### Frozen model-specific exact OLD cohorts'))
display(pd.DataFrame([{
    'model': 'RemoteCLIP', 'mapped_events': len(remoteclip_old_df), 'common_ready_events': cohort_audit.get('remoteclip_old_events', 0),
}, {
    'model': 'XGBoost', 'mapped_events': len(xgboost_old_df), 'common_ready_events': cohort_audit.get('xgboost_old_events', 0),
}]))
display(remoteclip_old_df.head(10))
display(xgboost_old_df.head(10))
profile_split_df = pd.DataFrame([{'profile': name, **values} for name, values in split_manifest.get('profiles', {}).items()])
display(Markdown('### Quick and final split manifests'))
display(profile_split_df)
display(Markdown('### Frozen final grouped folds'))
display(split_df)

### Cohort counts

,events,samples,logging,wildfire,event_review_events,legacy_manual_events,explicit_event_links,virtual_legacy_links,remoteclip_old_ready_events,xgboost_old_ready_events
0,502,232,265,237,138,364,4213,1025,135,127


### Excluded from the common model-ready cohort

,event_id,sample_id,event_year,human_label,xgboost_input_status,remoteclip_input_status
3,f1c8f33b62f33dadc280e198765c383e1a328485,03729ae3b0bec34e691900e592270d87a31d71db,2025,Wildfire,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
6,e9a33b242a09a3a2eb30da9034c6b03a56f43b3e,05d5126ea7bf1e23249a4315161430b5677f91c8,2025,Wildfire,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
7,ddb7a7d057d1fa51805ec2359e5160132f2ff891,0633c87245d4f0e672ff4dd3cc591d01615699b0,2016,Wildfire,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
9,8629394dee8b9c61dec5a8fc0865659fc51f33f4,08c6d954183aaf321ac8c73072d3cbda0ef5a6c9,2016,Logging,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
10,7a5939b1f9f6024cdd89d0a564f046eee396c65a,08c6d954183aaf321ac8c73072d3cbda0ef5a6c9,2017,Logging,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
...,...,...,...,...,...,...
487,155965d7d23828ad51396af79db1aaec5f78a081,f7fcaa4c0cff6ab5707f8434344e15afb8753521,2019,Wildfire,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
493,7086bcac729d9ffa11077bc64d4ee610208c8152,f9faec309f4637a592cdf85d139b1acf7416ec94,2019,Wildfire,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
494,a91cc7b6d04561d142fda6661840c4d903f18e9e,f9faec309f4637a592cdf85d139b1acf7416ec94,2020,Wildfire,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST
497,7cdaa256c600c199c13746a2ed1bc169391deed4,fe539ee1c173af056571974ba6e89ddb5a48b0ea,2019,Logging,MISSING_PRE_OR_POST,MISSING_PRE_OR_POST


### Duplicate audit

,duplicate_record_ids,duplicate_sample_year_rows
0,0,0


... (6 outputs left out)

### Quick and final split manifests

,profile,n_splits,seeds,split_manifest_hash
0,final,5,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]",b9cb126c8c6adbaba982b95b6f65ad6590188b52a2c462...
1,quick,5,[42],8e4794c20a888a57fd27a0ad296aaa5e95115edd74516e...


### Frozen final grouped folds

,requested_seed,effective_seed,seed,fold,train_events,train_samples,train_logging,train_wildfire,test_events,test_samples,test_logging,test_wildfire,group_overlap
0,0,0,0,1,213,177,86,127,58,45,23,35,0
1,0,0,0,2,218,176,88,130,53,46,21,32,0
2,0,0,0,3,217,177,87,130,54,45,22,32,0
3,0,0,0,4,217,181,87,130,54,41,22,32,0
4,0,0,0,5,219,177,88,131,52,45,21,31,0
5,1,1,1,1,218,177,88,130,53,45,21,32,0
6,1,1,1,2,217,176,88,129,54,46,21,33,0
7,1,1,1,3,213,179,85,128,58,43,24,34,0
8,1,1,1,4,217,175,87,130,54,47,22,32,0
9,1,1,1,5,219,181,88,131,52,41,21,31,0


## Cohort-size metrics

OLD and FULL rows use identical frozen test events. The primary metrics are Logging recall, balanced accuracy, and macro F1.

In [12]:
xgb_path = OUTPUT_ROOT / 'checkpoints' / 'T03_xgboost_v2' / 'summary_metrics.csv'
rc_path = OUTPUT_ROOT / 'checkpoints' / 'T04_remoteclip_v2' / 'summary_metrics.csv'
metrics_df = pd.concat([
    pd.read_csv(xgb_path) if xgb_path.exists() else pd.DataFrame(),
    pd.read_csv(rc_path) if rc_path.exists() else pd.DataFrame(),
], ignore_index=True)
metric_columns = ['model', 'cohort_mode', 'logging_recall_mean', 'balanced_accuracy_mean', 'macro_f1_mean', 'fold_runs']
confusion_columns = ['model', 'cohort_mode', 'tn_total', 'fp_total', 'fn_total', 'tp_total']
display(Markdown('### Aggregate metrics'))
display(metrics_df[metric_columns] if not metrics_df.empty else metrics_df)
display(Markdown('### Aggregate confusion matrices'))
display(metrics_df[confusion_columns] if not metrics_df.empty else metrics_df)

### Aggregate metrics

,model,cohort_mode,logging_recall_mean,balanced_accuracy_mean,macro_f1_mean,fold_runs
0,XGBOOST,FULL_COHORT_CV,0.718512,0.793652,0.796785,50
1,XGBOOST,OLD_COHORT_CV,0.655257,0.763171,0.766484,50
2,REMOTECLIP_LOGREG,FULL_COHORT_CV,0.978925,0.950369,0.943720,50
3,REMOTECLIP_LOGREG,OLD_COHORT_CV,0.887849,0.914678,0.916212,50


### Aggregate confusion matrices

,model,cohort_mode,tn_total,fp_total,fn_total,tp_total
0,XGBOOST,FULL_COHORT_CV,782,308,212,1408
1,XGBOOST,OLD_COHORT_CV,712,378,208,1412
2,REMOTECLIP_LOGREG,FULL_COHORT_CV,1067,23,127,1493
3,REMOTECLIP_LOGREG,OLD_COHORT_CV,968,122,95,1525


## Four-artifact smoke predictions

Each row contains independent predictions from XGBoost legacy/v2 and RemoteCLIP-LogReg legacy/v2.

In [13]:
smoke_path = OUTPUT_ROOT / 'checkpoints' / 'T05_four_artifact_smoke' / 'smoke_predictions.csv'
smoke_df = pd.read_csv(smoke_path) if smoke_path.exists() else pd.DataFrame()
display(smoke_df)

,event_id,sample_id,event_year,human_label,remoteclip_legacy_v1_prediction,remoteclip_legacy_v1_probability_logging,remoteclip_legacy_v1_probability_wildfire,remoteclip_v2_prediction,remoteclip_v2_probability_logging,remoteclip_v2_probability_wildfire,xgboost_legacy_v1_prediction,xgboost_legacy_v1_probability_logging,xgboost_legacy_v1_probability_wildfire,xgboost_v2_prediction,xgboost_v2_probability_logging,xgboost_v2_probability_wildfire
0,7a2033025cb30d007f309d8c947457ff57c535cc,0a1d1a56ea189abd64fe1ba2496178050a9a9c0f,2020,Logging,Logging,0.999946,0.000054,Logging,1.000000,8.620183e-09,Logging,0.892011,0.107989,Logging,0.921066,0.078934
1,7233ce5ad3a5bbdbcc13e990ee3cb6ea4ea190e6,12fb91c579cb98d698ed564cafab002a8735a4a7,2021,Logging,Logging,0.999909,0.000091,Logging,0.999998,2.114516e-06,Logging,0.981498,0.018502,Logging,0.975822,0.024178
2,9a35406b5b606c7044f340b55d4a380487695bd7,139f5bff37595aa2099e760de9088675b9d11310,2021,Logging,Wildfire,0.005253,0.994747,Logging,0.981197,1.880253e-02,Wildfire,0.014368,0.985632,Logging,0.705645,0.294355
3,a7698bb4fce40caa3d2f829c252fb4a124f98c7e,139f5bff37595aa2099e760de9088675b9d11310,2022,Logging,Wildfire,0.052805,0.947195,Logging,0.999865,1.354008e-04,Wildfire,0.014368,0.985632,Logging,0.709635,0.290365
4,9e8f8d68ce5f4f35595c51d17be23f0e3554b54f,1460cd7355fb4dfb878677df9787f4c7fef8052f,2025,Logging,Logging,0.999363,0.000637,Logging,0.999988,1.203778e-05,Logging,0.921124,0.078876,Logging,0.976715,0.023285
5,f0ea8245f0e531ddf93a9efbd9dfdfd70c78d29e,014ebcfa8c4c4e37669da9360bb2f7d33ecad401,2021,Wildfire,Wildfire,0.001151,0.998849,Wildfire,0.007374,9.926262e-01,Wildfire,0.097097,0.902903,Wildfire,0.085165,0.914835
6,80461db9d77577a5dcad4a6a1e6888ed22fa7896,0219f58cd6ca95c2f46aa5a02902499dc907b131,2024,Wildfire,Wildfire,0.000051,0.999949,Wildfire,0.000137,9.998634e-01,Wildfire,0.019173,0.980827,Wildfire,0.009640,0.990360
7,9db54f6a5d220c39ea506e4bafd413b09b083779,03729ae3b0bec34e691900e592270d87a31d71db,2024,Wildfire,Wildfire,0.000126,0.999874,Wildfire,0.000038,9.999615e-01,Wildfire,0.019173,0.980827,Wildfire,0.015135,0.984865
8,201a165e43eccd77c082f393e7fda3414de66eee,042d302a862f287b13318d16ad25798636d851e0,2020,Wildfire,Wildfire,0.014077,0.985923,Wildfire,0.002581,9.974185e-01,Wildfire,0.019173,0.980827,Wildfire,0.009956,0.990044
9,99c9f3e316febfea8141f579cbb8886e2494d2a8,05d5126ea7bf1e23249a4315161430b5677f91c8,2024,Wildfire,Wildfire,0.000002,0.999998,Wildfire,0.000082,9.999180e-01,Wildfire,0.043628,0.956372,Wildfire,0.035845,0.964155
